# Load test results: latency vs concurrency and slot count

Analyses the `slots.N.csv` files produced by `tests/load_test.py`.

Two plots:
1. **Latency vs concurrency** — mean ± SEM and p95, one line per slot count.
2. **Latency at concurrency = 8 vs slot count** — mean ± SEM and p95.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

RESULTS_DIR = Path('..') / 'tests' / 'results'

In [ ]:
# ---------------------------------------------------------------------------
# Load all slots.N.csv files present in the results directory
# ---------------------------------------------------------------------------
slot_files = sorted(
    RESULTS_DIR.glob('slots.*.csv'),
    key=lambda p: int(re.search(r'slots\.(\d+)\.csv', p.name).group(1)),
)

if not slot_files:
    raise FileNotFoundError(f'No slots.*.csv files found in {RESULTS_DIR.resolve()}')

frames = []
for path in slot_files:
    df = pd.read_csv(path)
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

# Keep only successful rows
data = raw[raw['error'] == 'NaN'].copy()
data['slots'] = data['slots'].astype(int)
data['concurrency'] = data['concurrency'].astype(int)
data['latency_s'] = data['latency_s'].astype(float)

slot_counts = sorted(data['slots'].unique())
print(f'Slot counts found: {slot_counts}')
print(f'Concurrency levels: {sorted(data["concurrency"].unique())}')
print(f'Total successful rows: {len(data)}')

In [ ]:
# ---------------------------------------------------------------------------
# Helper: compute mean, SEM, p95 grouped by specified columns
# ---------------------------------------------------------------------------
def summarise(df, group_cols):
    agg = (
        df.groupby(group_cols)['latency_s']
        .agg(
            mean='mean',
            count='count',
            std='std',
            p95=lambda x: x.quantile(0.95),
        )
        .reset_index()
    )
    agg['sem'] = agg['std'] / np.sqrt(agg['count'])
    return agg

## Plot 1: latency vs concurrency, by slot count

Each slot count gets a pair of traces: solid line for mean ± SEM, dotted line with diamond markers for p95.

In [ ]:
stats_by_slots = summarise(data, ['slots', 'concurrency'])

colours = [
    '#636EFA', '#EF553B', '#00CC96', '#AB63FA',
    '#FFA15A', '#19D3F3', '#FF6692', '#B6E880',
]

fig1 = go.Figure()

for i, n_slots in enumerate(slot_counts):
    subset = stats_by_slots[stats_by_slots['slots'] == n_slots].sort_values('concurrency')
    colour = colours[i % len(colours)]
    label = f'{n_slots} slot{"s" if n_slots > 1 else ""}'

    # Mean ± SEM
    fig1.add_trace(go.Scatter(
        x=subset['concurrency'],
        y=subset['mean'],
        error_y=dict(type='data', array=subset['sem'].tolist(), visible=True),
        mode='lines+markers',
        line=dict(color=colour, width=1.5, dash='solid'),
        marker=dict(size=8),
        name=f'{label} — mean ± SEM',
        legendgroup=str(n_slots),
    ))

    # p95
    fig1.add_trace(go.Scatter(
        x=subset['concurrency'],
        y=subset['p95'],
        mode='lines+markers',
        line=dict(color=colour, width=1.5, dash='dot'),
        marker=dict(size=7, symbol='diamond'),
        name=f'{label} — p95',
        legendgroup=str(n_slots),
    ))

all_concurrencies = sorted(data['concurrency'].unique())

fig1.update_layout(
    title='Latency vs concurrency by slot count',
    xaxis_title='Concurrency (simultaneous requests)',
    yaxis_title='Latency (s)',
    xaxis=dict(tickmode='array', tickvals=all_concurrencies),
    legend=dict(groupclick='toggleitem'),
    margin=dict(l=50, r=20, t=50, b=50),
)

fig1.show()

## Plot 2: latency at concurrency = 8 vs slot count

Shows how increasing the number of server slots affects latency when 8 requests arrive simultaneously.

In [ ]:
TARGET_CONCURRENCY = 8

data_c8 = data[data['concurrency'] == TARGET_CONCURRENCY]

if data_c8.empty:
    print(f'No data for concurrency={TARGET_CONCURRENCY}. Adjust TARGET_CONCURRENCY above.')
else:
    stats_c8 = summarise(data_c8, ['slots'])

    fig2 = go.Figure()

    # Mean ± SEM
    fig2.add_trace(go.Scatter(
        x=stats_c8['slots'],
        y=stats_c8['mean'],
        error_y=dict(type='data', array=stats_c8['sem'].tolist(), visible=True),
        mode='lines+markers',
        line=dict(width=1.5, dash='solid'),
        marker=dict(size=9),
        name='Mean ± SEM',
    ))

    # p95
    fig2.add_trace(go.Scatter(
        x=stats_c8['slots'],
        y=stats_c8['p95'],
        mode='lines+markers',
        line=dict(width=1.5, dash='dot'),
        marker=dict(size=8, symbol='diamond'),
        name='p95',
    ))

    fig2.update_layout(
        title=f'Latency at concurrency = {TARGET_CONCURRENCY} vs slot count',
        xaxis_title='Server slots (--parallel)',
        yaxis_title='Latency (s)',
        xaxis=dict(tickmode='array', tickvals=stats_c8['slots'].tolist()),
        margin=dict(l=50, r=20, t=50, b=50),
    )

    fig2.show()